# 🌌 Space Odyssey — **Extended** GRPO Training Notebook

This is the **longer-training variant** of `Space_Odyssey_Colab.ipynb`.
Use this **only after** the canonical notebook works end-to-end.

**What's different from the canonical notebook**
- Episode counts bumped: **35 / 75 / 40 = 150 episodes** (canonical: 20 / 50 / 30 = 100).
- Output adapter dir: `overseer_grpo_extended/` (so it doesn't overwrite `overseer_grpo_final/`).
- Estimated runtime: **~60-75 min on a free T4** (canonical: ~35 min).

**Same hard guardrails as canonical (do not change):**
- `num_generations` stays at **4 / 4 / 2** in `training/grpo_train.py` — raising past 4 OOMs on T4.
- `max_completion_length = 128` — bumping past ~256 with 4 generations OOMs on T4.

**CRITICAL — Follow exactly:**
1. Set runtime → **GPU**: Runtime → Change runtime type → T4 GPU
2. Run **Cell 1** only
3. ⚠️ **RESTART**: Runtime → Restart session
4. Run **Cells 2, 3, 4, 5, 6, 7, 8** in order

**Do NOT skip the restart.** That is what causes the 'deps missing' error.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL ONLY
# After this finishes → Runtime → Restart session → then run Cell 2+
# ══════════════════════════════════════════════════════════════════════

!pip install --upgrade pip -q

# Install Unsloth (the fast training engine)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install the rest AFTER unsloth (avoids version fights)
!pip install trl peft accelerate bitsandbytes -q
!pip install transformers datasets tokenizers -q
!pip install gymnasium matplotlib plotly -q

print()
print('=' * 60)
print('✅ Installation done!')
print()
print('>>> NOW DO: Runtime → Restart session <<<')
print('>>> Then run Cells 2, 3, 4, 5, 6, 7, 8 in order <<<')
print('=' * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — CLONE REPO (Run after restarting runtime)
# ══════════════════════════════════════════════════════════════════════

import os

if not os.path.exists('/content/Space-Odyssey'):
    !git clone https://github.com/lokendra005/Space-Odyssey.git /content/Space-Odyssey
else:
    !cd /content/Space-Odyssey && git pull origin main
    print('Already cloned, pulled latest.')

%cd /content/Space-Odyssey
print('\n✅ Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — DIAGNOSTIC CHECK
# Run this to see EXACTLY what is installed and available
# ══════════════════════════════════════════════════════════════════════

import importlib, sys

REQUIRED = ['torch', 'unsloth', 'trl', 'peft', 'transformers', 'datasets', 'bitsandbytes', 'accelerate']

all_ok = True
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'  ✅ {pkg:<18} {ver}')
    except ImportError as e:
        print(f'  ❌ {pkg:<18} MISSING — {e}')
        all_ok = False

import torch
print(f'\n  GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ NOT FOUND"}')
print(f'  VRAM: {round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else 0} GB')

if not all_ok:
    print('\n⚠️  Some packages are missing!')
    print('   Go back, run Cell 1, then RESTART RUNTIME and try again.')
else:
    print('\n✅ All packages ready! Proceed to Cell 4.')

if not torch.cuda.is_available():
    print('⚠️  NO GPU DETECTED — training will be extremely slow or fail.')
    print('   Go to Runtime → Change runtime type → GPU (T4)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — ENVIRONMENT SANITY CHECK
# ══════════════════════════════════════════════════════════════════════

from environment.station_env import ProcurementDriftEnv
from training.reward import is_proposal_dangerous, compute_violation_severity

env = ProcurementDriftEnv()
obs, info = env.reset(seed=42)
state = env._flat_state()
proposal = env.current_proposal

print('--- Station Telemetry (seed=42) ---')
print(f'  O2   = {state["oxygen"]:.1f}%')
print(f'  Power= {state["power"]:.1f}%')
print(f'  Hull = {state["hull_integrity"]:.1f}%')
print(f'  Proposal type   : {proposal["type"]}')
print(f'  Label (risk_lv) : {proposal["risk_level"]}')
print(f'  Oracle danger?  : {is_proposal_dangerous(state, proposal)}')
print(f'  Severity score  : {compute_violation_severity(state, proposal):.3f}')
env.close()
print('\n✅ Environment healthy!')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — PHASE 1: SFT REASONING WARMUP (~8 min on T4)
# Teaches the model to write ANALYSIS → DECISION → REASON
# ══════════════════════════════════════════════════════════════════════

# Must import unsloth before everything else
import unsloth

from training.sft_warmup import _HAS_TRAIN_DEPS, _TRAIN_DEPS_ERROR, run_sft

if not _HAS_TRAIN_DEPS:
    raise RuntimeError(
        f'\n❌ Missing training dependency: {_TRAIN_DEPS_ERROR}\n'
        'Fix: Go back to Cell 1, run it, then RESTART RUNTIME (Runtime → Restart session),\n'
        'then run from Cell 2 again.'
    )

print('🧠 Starting Phase 1: Reasoning Chain Warmup...')
model, tokenizer = run_sft()
print('\n✅ Phase 1 done! → overseer_lora_warmup/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — PHASES 2-4: EXTENDED CURRICULUM GRPO (~45-60 min on T4)
# Slightly longer than canonical, but tuned to NOT trigger Unsloth's
# "smartly offload gradients to save VRAM" path that murders throughput.
# Watch VPR rise toward 95%+ = model is learning!
# ══════════════════════════════════════════════════════════════════════

from training.grpo_train import run_grpo_training

# T4 (16 GB) — VRAM-conservative defaults from training/grpo_train.py:
#   num_generations         = 2  (minimum for GRPO advantage)
#   gradient_accumulation   = 4
#   max_prompt_length       = 320
#   max_completion_length   = 96
# Together these keep peak VRAM under ~10 GB so Unsloth does NOT need to
# CPU-offload gradients — that's the only thing that makes runs take 5h+.

print('🛡️ Starting EXTENDED GRPO Reinforcement Learning...')
print('   Phase 1 (Easy)     :  25 episodes — no adversaries, exploration')
print('   Phase 2 (Hard)     :  50 episodes — adversaries on, deception detection')
print('   Phase 3 (Precision):  25 episodes — calibrate against false alarms')
print('   Total              : 100 episodes (~45-60 min on T4)')
print('   (If this completes comfortably, raise to 35/75/40 next session.)')
print()

run_grpo_training(
    sft_adapter_path   = 'overseer_lora_warmup',
    output_dir         = 'overseer_grpo_extended',   # separate from canonical
    easy_episodes      = 25,
    hard_episodes      = 50,
    precision_episodes = 25,
)

print('\n🏆 EXTENDED TRAINING COMPLETE! → overseer_grpo_extended/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — EVALUATION & PLOTS
# Reads the training log written during Cell 6 and renders curves.
# ══════════════════════════════════════════════════════════════════════

import os
os.makedirs('assets', exist_ok=True)

!python eval/evaluate.py
!python eval/plot_training_curves.py

from IPython.display import Image, display
plots = [
    ('assets/training_curve.png',       'Training Reward Curve'),
    ('assets/violation_prevention.png', 'Violation Prevention Rate'),
    ('assets/eval_results.png',         'Trained vs Baseline'),
]
for path, title in plots:
    if os.path.exists(path):
        print(f'\n── {title} ──')
        display(Image(path))
    else:
        print(f'[skip] {path} not generated')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8 — DOWNLOAD TRAINED BRAIN (extended)
# ══════════════════════════════════════════════════════════════════════

import shutil, os

if os.path.isdir('overseer_grpo_extended'):
    shutil.make_archive('trained_brain_extended', 'zip', '.', 'overseer_grpo_extended')
    size_mb = round(os.path.getsize('trained_brain_extended.zip') / 1e6, 1)
    print(f'✅ Zipped → trained_brain_extended.zip ({size_mb} MB)')
    from google.colab import files
    files.download('trained_brain_extended.zip')
else:
    print('⚠️ overseer_grpo_extended/ not found — did Cell 6 complete successfully?')